In [ ]:
# Mount Google Drive to save/load model
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install required libraries
!pip install torch tokenizers accelerate -U
!pip install rouge_score  # For evaluation
!pip install torch tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 790.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
import math
import os
from rouge_score import rouge_scorer

# Step 1: Load and prepare the dataset
df = pd.read_csv('/content/drive/MyDrive/Transformer/mtsamples.csv')
df = df[['transcription', 'description']].dropna()
df = df.rename(columns={'transcription': 'text', 'description': 'summary'})

# Step 2: Train a WordPiece tokenizer on the dataset
texts = list(df['text']) + list(df['summary'])
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = WordPieceTrainer(vocab_size=30000, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"])
tokenizer.train_from_iterator(texts, trainer=trainer)

# Save tokenizer
tokenizer_path = "/content/drive/MyDrive/Transformer"
os.makedirs(tokenizer_path, exist_ok=True)
tokenizer.save(os.path.join(tokenizer_path, "tokenizer.json"))

# Encode dataset
def encode_texts(texts, summaries, max_input_length=512, max_target_length=128):
    input_encodings = []
    target_encodings = []
    for text, summary in zip(texts, summaries):
        input_ids = tokenizer.encode(text).ids
        target_ids = tokenizer.encode(summary).ids
        if len(input_ids) > max_input_length:
            input_ids = input_ids[:max_input_length]
        if len(target_ids) > max_target_length:
            target_ids = target_ids[:max_target_length]
        input_encodings.append(input_ids)
        target_encodings.append(target_ids)
    return input_encodings, target_encodings

input_encodings, target_encodings = encode_texts(df['text'], df['summary'])

# Pad sequences
def pad_sequences(sequences, max_length, pad_token_id):
    padded = []
    for seq in sequences:
        if len(seq) < max_length:
            seq = seq + [pad_token_id] * (max_length - len(seq))
        padded.append(seq[:max_length])
    return padded

max_input_length = 512
max_target_length = 128
pad_token_id = tokenizer.token_to_id("[PAD]")
input_encodings = pad_sequences(input_encodings, max_input_length, pad_token_id)
target_encodings = pad_sequences(target_encodings, max_target_length, pad_token_id)

# Convert to tensors
input_tensors = torch.tensor(input_encodings, dtype=torch.long)
target_tensors = torch.tensor(target_encodings, dtype=torch.long)

# Split into train and validation (80-20)
train_size = int(0.8 * len(input_tensors))
train_inputs, val_inputs = input_tensors[:train_size], input_tensors[train_size:]
train_targets, val_targets = target_tensors[:train_size], target_tensors[train_size:]

# Step 3: Define Transformer model from scratch
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src = self.embedding(src) * math.sqrt(self.d_model)
        tgt = self.embedding(tgt) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)
        output = self.transformer(src, tgt, src_mask, tgt_mask)
        output = self.fc_out(output)
        return output

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

# Instantiate model
vocab_size = tokenizer.get_vocab_size()
model = TransformerModel(vocab_size=vocab_size).cuda()

# Step 4: Training setup
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=pad_token_id)
scaler = GradScaler()

# Create dataset and dataloader
class MedicalDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {'input_ids': self.inputs[idx], 'labels': self.targets[idx]}

train_dataset = MedicalDataset(train_inputs, train_targets)
val_dataset = MedicalDataset(val_inputs, val_targets)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4)

# Step 5: Training loop
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        src = batch['input_ids'].cuda()
        tgt = batch['labels'].cuda()
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).cuda()

        with autocast():
            output = model(src, tgt_input, tgt_mask=tgt_mask)
            loss = criterion(output.reshape(-1, vocab_size), tgt_output.reshape(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            src = batch['input_ids'].cuda()
            tgt = batch['labels'].cuda()
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]
            tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).cuda()

            with autocast():
                output = model(src, tgt_input, tgt_mask=tgt_mask)
                loss = criterion(output.reshape(-1, vocab_size), tgt_output.reshape(-1))
            total_loss += loss.item()

            # Decode for ROUGE
            preds = torch.argmax(output, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(tgt_output.cpu().numpy())
    return total_loss / len(loader), all_preds, all_labels

# Step 6: Train the model
num_epochs = 30  # More epochs due to training from scratch
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)

    # Compute ROUGE scores
    decoded_preds = [tokenizer.decode(pred, skip_special_tokens=True) for pred in val_preds]
    decoded_labels = [tokenizer.decode(label, skip_special_tokens=True) for label in val_labels]
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(label, pred) for pred, label in zip(decoded_preds, decoded_labels)]
    rouge1 = sum(score['rouge1'].fmeasure for score in rouge_scores) / len(rouge_scores)

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, ROUGE-1: {rouge1:.4f}")

# Step 7: Save the model
model_path = "/content/drive/MyDrive/Transformer"
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to: {model_path}")

/tmp/ipython-input-4193130313.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-4193130313.py:156: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-4193130313.py:178: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1, Train Loss: 6.7389, Val Loss: 6.0394, ROUGE-1: 0.1594
Epoch 2, Train Loss: 5.4838, Val Loss: 5.3924, ROUGE-1: 0.1895
Epoch 3, Train Loss: 4.7665, Val Loss: 4.9051, ROUGE-1: 0.1689
Epoch 4, Train Loss: 4.1942, Val Loss: 4.5177, ROUGE-1: 0.2152
Epoch 5, Train Loss: 3.6936, Val Loss: 4.1553, ROUGE-1: 0.1724
Epoch 6, Train Loss: 3.2391, Val Loss: 3.8263, ROUGE-1: 0.2582
Epoch 7, Train Loss: 2.8137, Val Loss: 3.4976, ROUGE-1: 0.2463
Epoch 8, Train Loss: 2.4272, Val Loss: 3.2177, ROUGE-1: 0.2285
Epoch 9, Train Loss: 2.0613, Val Loss: 2.9594, ROUGE-1: 0.2733
Epoch 10, Train Loss: 1.7543, Val Loss: 2.6839, ROUGE-1: 0.2976
Epoch 11, Train Loss: 1.4668, Val Loss: 2.4353, ROUGE-1: 0.3854
Epoch 12, Train Loss: 1.2387, Val Loss: 2.2166, ROUGE-1: 0.3432
Epoch 13, Train Loss: 1.0515, Val Loss: 2.0342, ROUGE-1: 0.5252
Epoch 14, Train Loss: 0.8718, Val Loss: 1.8536, ROUGE-1: 0.3287
Epoch 15, Train Loss: 0.7507, Val Loss: 1.7251, ROUGE-1: 0.3973
Epoch 16, Train Loss: 0.6383, Val Loss: 1.5869, R

In [ ]:
import torch
from tokenizers import Tokenizer
import os
from IPython.display import display, HTML
from ipywidgets import widgets

# Load tokenizer
tokenizer_path = "/content/drive/MyDrive/Transformer/tokenizer.json"
tokenizer = Tokenizer.from_file(tokenizer_path)

# Define Transformer model (same as training)
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(torch.nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = torch.nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc_out = torch.nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src = self.embedding(src) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float))
        tgt = self.embedding(tgt) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float))
        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)
        output = self.transformer(src, tgt, src_mask, tgt_mask)
        output = self.fc_out(output)
        return output

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask

# Load model
model_path = "/content/drive/MyDrive/Transformer/model.pt"
vocab_size = tokenizer.get_vocab_size()
model = TransformerModel(vocab_size=vocab_size).cuda()
model.load_state_dict(torch.load(model_path))
model.eval()

# Function to generate summary
def generate_summary(text, max_length=128):
    # Encode input
    input_ids = tokenizer.encode(text).ids
    input_ids = input_ids[:512]  # Truncate to max input length
    input_ids = input_ids + [tokenizer.token_to_id("[PAD]")] * (512 - len(input_ids))
    input_tensor = torch.tensor([input_ids], dtype=torch.long).cuda()

    # Generate output
    generated_ids = [tokenizer.token_to_id("[CLS]")]
    for _ in range(max_length):
        tgt_tensor = torch.tensor([generated_ids], dtype=torch.long).cuda()
        tgt_mask = model.generate_square_subsequent_mask(tgt_tensor.size(1)).cuda()
        with torch.no_grad():
            output = model(input_tensor, tgt_tensor, tgt_mask=tgt_mask)
        next_token = torch.argmax(output[:, -1, :], dim=-1).item()
        generated_ids.append(next_token)
        if next_token == tokenizer.token_to_id("[SEP]"):
            break

    # Decode output
    summary = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return summary

# Option 1: Colab Form Input (recommended for presentation)
text_input = widgets.Textarea(
    value='',
    placeholder='Paste or type your medical transcript here...',
    description='Input Text:',
    layout={'width': '600px', 'height': '200px'}
)
output_label = widgets.Label(value="Generated Summary: ")
output_text = widgets.Textarea(
    value='',
    disabled=True,
    layout={'width': '600px', 'height': '100px'}
)
button = widgets.Button(description="Generate Summary")

def on_button_clicked(b):
    if text_input.value.strip():
        summary = generate_summary(text_input.value)
        output_text.value = summary
    else:
        output_text.value = "Please enter some text to summarize."

button.on_click(on_button_clicked)

# Display form
display(text_input, button, output_label, output_text)

Textarea(value='', description='Input Text:', layout=Layout(height='200px', width='600px'), placeholder='Paste…

Button(description='Generate Summary', style=ButtonStyle())

Label(value='Generated Summary: ')

Textarea(value='', disabled=True, layout=Layout(height='100px', width='600px'))